## Install packages

In [1]:
# 1. Reinstall the core without hard-pinning every version
!pip install "transformers>=4.48.0" "huggingface_hub>=0.28.0" "pydantic>=2.0" "gradio>=5.0.0" "accelerate>=1.0.0" "bitsandbytes>=0.45.0" -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
langchain-core 0.3.79 requires packaging<26.0.0,>=23.2.0, but you have packaging 26.0rc2 which is incompatible.
fastai 2.8.4 requires fastcore<1.9,>=1.8.0, but you have fastcore 1.11.3 which is incompatible.


In [2]:
# 2. Verify
import pydantic
import transformers
import gradio as gr
import asyncio

print(f"✅ Recovery Successful!")
print(f"Pydantic: {pydantic.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Gradio: {gr.__version__}")

✅ Recovery Successful!
Pydantic: 2.11.10
Transformers: 4.57.1
Gradio: 5.49.1


## Kaggle secret

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# 1. Get the token from the "Safe" (Secrets)
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

# 2. Log in to Hugging Face
login(token=secret_value_0)


In [4]:
#import bitsandbytes as bnb
import torch
import numpy as np 
import pandas as pd 
import json
import copy
import re
import os
import gc
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, AutoTokenizer, AutoModelForCausalLM
from PIL import Image


#print(f"✅ BitsAndBytes version: {bnb.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
print(f"✅ GPU Count: {torch.cuda.device_count()}")

2026-03-08 07:07:45.244945: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772953665.634085      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772953665.740491      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772953666.707794      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772953666.707832      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772953666.707835      24 computation_placer.cc:177] computation placer alr

✅ CUDA Available: True
✅ GPU Count: 2


## Memory manager

In [5]:
class MemoryManager:
    @staticmethod
    def clear():
        """Forcefully clears the GPU cache and garbage collects."""
        # 1. Clear PyTorch's internal cache
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        
        # 2. Force Python's garbage collector
        gc.collect()
        
        # 3. Log current state (Optional, for debugging)
        # print(f"--- 🧹 Memory Scrubbed | Allocated: {torch.cuda.memory_allocated(0)/1024**2:.1f}MB ---")

## Load MedGemma (multi-modal and text)

In [6]:
'''
# 1. Create the Quantization Config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)
'''

# 2. Load Vision Specialist (GPU 0)
model_id_vision = "google/medgemma-1.5-4b-it"

# 1. The Brain (distributed across both GPUs)
model_vision = AutoModelForImageTextToText.from_pretrained(
    model_id_vision,
    torch_dtype=torch.bfloat16,
    device_map="auto", 
    token=secret_value_0
)

# 2. The Translator (lives in CPU RAM, manages the 'Max Context' prep)
processor_vision = AutoProcessor.from_pretrained(
    model_id_vision, 
    token=secret_value_0
)

'''
# 3. Load Text Critic/Scribe (GPU 1)
model_id_text = "google/gemma-2-2b-it"

tokenizer_text = AutoTokenizer.from_pretrained(model_id_text, token=secret_value_0)

model_text = AutoModelForCausalLM.from_pretrained(
    model_id_text,
    quantization_config=quant_config,
    device_map={"": 1}, # Force entirely onto second GPU
    token=secret_value_0
)
'''

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

'\n# 3. Load Text Critic/Scribe (GPU 1)\nmodel_id_text = "google/gemma-2-2b-it"\n\ntokenizer_text = AutoTokenizer.from_pretrained(model_id_text, token=secret_value_0)\n\nmodel_text = AutoModelForCausalLM.from_pretrained(\n    model_id_text,\n    quantization_config=quant_config,\n    device_map={"": 1}, # Force entirely onto second GPU\n    token=secret_value_0\n)\n'

In [7]:
def call_medgemma_universal(prompt, image_path=None):
    try:
        # 1. Image handling
        image = None
        content = []
        if image_path:
            image = Image.open(image_path).convert("RGB")
            content.append({"type": "image"})
        
        content.append({"type": "text", "text": prompt})
        
        # 2. Template wrapping
        messages = [{"role": "user", "content": content}]
        formatted_prompt = processor_vision.apply_chat_template(messages, add_generation_prompt=True)

        # 3. Process inputs
        inputs = processor_vision(
            text=formatted_prompt, 
            images=image, 
            return_tensors="pt"
        )
        
        # Move inputs to the specific device 
        # where the model's INPUT embeddings live.
        inputs = {k: v.to(model_vision.device) for k, v in inputs.items()}

        # 4. Generate
        with torch.no_grad():
            output_tokens = model_vision.generate(
                **inputs,
                max_new_tokens=1024,
                do_sample=False, # Stable greedy decoding
                repetition_penalty=1.1,
                pad_token_id=processor_vision.tokenizer.eos_token_id
            )

        # 5. Decode
        prompt_length = inputs['input_ids'].shape[1]
        return processor_vision.decode(output_tokens[0][prompt_length:], skip_special_tokens=True)

    except Exception as e:
        return f"ERROR: {str(e)}"

## Prompt Library

In [8]:

IMAGING_SPECIALIST_PROMPT = """<image>
SYSTEM ROLE: Technical Vision Perception Engine.
TASK: Extract objective visual features.

STRICT CONSTRAINTS:
1. OUTPUT MUST BE A SINGLE JSON OBJECT.
2. DO NOT OUTPUT A LIST [].
3. DO NOT USE KEY NAMES OTHER THAN THOSE IN THE SCHEMA.

SCHEMA TO FOLLOW:
{
    "imaging_modality": "string (e.g., CXR|X-Ray|CT Head|MRI Brain|CT Abdomen|US|MRI)",
    "key_findings": ["string"],
    "laterality": "Left|Right|Bilateral",
    "critical_finding_detected": true|false,
    "confidence_level": "medium",
    "technical_quality": "adequate",
    "safety_note": "Visual feature extraction only."
}

FINAL COMMAND: Return ONLY the JSON object. Start your response with '{'.
"""


CLINICAL_SCRIBE_PROMPT_COMPREHENSIVE = """
ROLE: Expert Medical Scribe.
TASK: Synthesize 'MESSY CLINICIAN NOTES' and 'IMAGING FINDINGS' into a professional structured SOAP JSON note.

INPUT DATA:
1. MESSY NOTES: {messy_doctor_notes}
2. IMAGING FINDINGS: {imaging_findings_input}

CONSTRAINTS:
- Resolve abbreviations (e.g., 'pt c/o' -> 'Patient complains of').
- If 'IMAGING FINDINGS' are 'None', leave the 'Objective' section text-based only.
- The 'plan.management' field must be an array of objects (category, name, dose, etc.).
- Every vital sign must include a 'status' key (normal|high|low).
- If PII (names, phone numbers) is detected, redact it in text fields and set 'pii_detected': true.
- OUTPUT ONLY THE JSON OBJECT.

JSON SCHEMA:
{{
  "patient_summary": {{ "demographics": [], "critical_alerts": [] }},
  "subjective": {{ "chief_complaint": "", "hpi": [], "past_medical_history": [], "current_medications": [] }},
  "objective": {{ "vitals": [], "physical_exam": [], "imaging_findings": {{}} }},
  "assessment": {{ "clinical_synthesis": "", "primary_diagnosis": "", "differential_diagnosis": [] }},
  "plan": {{ "management": [], "monitoring": "", "patient_instructions": "" }},
  "safety_audit": {{ "status": "CLEAN", "pii_detected": false, "audit_findings": [], "confidence_score": 0.0, "action_required": "proceed" }}
}}
"""

CoT_AUDITOR_PROMPT_COMPREHENSIVE = """
ROLE: Senior Forensic Clinical Auditor & Privacy Guardian.
TASK: Conduct a rigorous structural, clinical, and safety audit comparing the 'RAW EVIDENCE' against the 'PROPOSED SOAP JSON'.

[RAW EVIDENCE]
- Messy Doctor Note: {messy_doctor_notes}
- Imaging Findings: {visual_findings}

[PROPOSED SOAP JSON]
{soap_json}

AUDIT PROTOCOL (Chain of Thought):
Step 1: PRIVACY SCAN (Category D)
- Scan the RAW EVIDENCE for any unmasked Personally Identifiable Information (PII) such as patient names, phone numbers, or exact addresses. Check if the SOAP JSON successfully caught it.

Step 2: CONTRADICTION & SAFETY SCAN (Category A)
- Cross-reference documented Allergies in the JSON against the prescribed Medications in the Plan.
- Verify Laterality (Left vs. Right) matches exactly between Evidence and SOAP.
- Ensure Objective Vitals match the Subjective description.

Step 3: OMISSION SCAN (Category B)
- Check for "Ignored Red Alerts" (e.g., abnormal vitals with no interventions in the Plan).
- Ensure chronic diseases mentioned in the Evidence are NOT missing from the Past Medical History.

Step 4: HALLUCINATION SCAN (Category C)
- Identify any physical exam findings, specific demographic details, or subjective quotes in the SOAP JSON that DO NOT exist in the RAW EVIDENCE.

Step 5: CLINICAL COHERENCE SCAN (The Diagnostic Link)
- Evaluate if the 'assessment.primary_diagnosis' is logically supported by the 'subjective' and 'objective' data.
- Ensure the 'plan.management' directly and appropriately treats the primary diagnosis. Flag any dangerous or nonsensical treatments.

OUTPUT FORMAT:
Reasoning: 
1. Privacy Findings: [Notes on PII]
2. Contradiction/Safety Findings: [Notes on Allergies, Laterality, Vitals]
3. Omission Findings: [Notes on missing PMH or absent treatment plans]
4. Hallucination Findings: [Notes on fabricated details]
5. Clinical Coherence: [Notes on S+O -> A -> P logic]

JSON: 
{{
  "safety_audit": {{
    "status": "CLEAN" | "FLAGGED",
    "pii_detected": true | false,
    "contradictions": [
      "Specific detail of the error (e.g., 'Plan prescribes Bactrim, but patient has Sulfa allergy.', 'Diagnosis of Pneumonia is unsupported by clear lung exam.')"
    ],
    "confidence_score": 0.0 to 1.0,
    "action_required": "proceed" | "repair"
  }}
}}

CRITICAL: If ANY error, omission, hallucination, incoherence, or PII is found, "status" MUST be "FLAGGED" and "action_required" MUST be "repair" or "human_escalation". If no errors exist, output "status": "CLEAN" with an empty contradictions array.
"""


MISTAKE_EDITOR_PROMPT = """
SYSTEM: Your previous SOAP note was REJECTED for clinical inaccuracies.

REJECTED DRAFT:
{rejected_soap}

ERRORS DETECTED BY AUDITOR:
{audit_findings_str}

ORIGINAL CLINICAL EVIDENCE:
- Vision Data: {vision_json_str}
- Patient Notes: {symptoms_text}

TASK: Rewrite the SOAP note. 
- Keep the parts that were correct. 
- SURGICALLY FIX the errors listed above. 
- DO NOT repeat the specific mistakes found in the Rejected Draft.

OUTPUT: Return ONLY the corrected JSON.
"""

## PII & Safety Checks

In [9]:
def pii_deidentification_gate(text):
    """
    Acts as a 'De-identification Gate' that redacts PII before 
    it reaches the LLM agents[cite: 34, 51, 93].
    """
    clean_text = text
    
    # 1. Patterns to Redact (Demonstrating Production Logic)
    patterns = {
        r'\b\d{3}-\d{3}-\d{4}\b': '[PHONE_NUMBER]', # US/Generic Phone
        r'\b[A-Z]{2}\d{6}[A-Z]\b': '[NHS_NUMBER]',  # Simulated NHS Pattern 
        r'\b\d{3}-\d{2}-\d{4}\b': '[SSN]',          # Social Security
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b': '[EMAIL]' # 
    }
    
    # 2. Keywords to Scrub
    sensitive_keywords = ["NHS", "Social Security", "Phone", "Address"]
    
    # Apply Pattern Scrubbing
    for pattern, placeholder in patterns.items():
        clean_text = re.sub(pattern, placeholder, clean_text)
        
    # Apply Keyword Scrubbing
    for word in sensitive_keywords:
        # Case-insensitive replacement
        clean_text = re.compile(re.escape(word), re.IGNORECASE).sub(f'[{word.upper()}_ID]', clean_text)
        
    # Senior Move: Log that a scrub occurred (Auditable Guarantee) [cite: 53, 95]
    if clean_text != text:
        print("🛡️ Safety Gate: PII detected and redacted.")
        
    return True, clean_text

## Validation & Repair Loop

In [10]:
import jsonschema
from jsonschema import validate

# This schema defines a valid SOAP note
SOAP_SCHEMA = {
    "type": "object",
    "properties": {
        "subjective": {
            "anyOf": [{"type": "string"}, {"type": "array", "items": {"type": "string"}}]
        },
        "objective": {
            "anyOf": [{"type": "string"}, {"type": "array", "items": {"type": "string"}}]
        },
        "assessment": {
            "anyOf": [{"type": "string"}, {"type": "array", "items": {"type": "string"}}]
        },
        "plan": {
            "anyOf": [{"type": "string"}, {"type": "array", "items": {"type": "string"}}]
        },
        "patient_status": {"enum": ["Stable", "Critical", "Unknown"]},
        "risk_score": {
            "anyOf": [
                {"type": "integer", "minimum": 1, "maximum": 10},
                {"type": "string", "pattern": "^([1-9]|10)$"} 
            ]
        }
    },
    "required": ["subjective", "objective", "assessment", "plan", "patient_status"]
}


VISION_SCHEMA = {
    "type": "object",
    "properties": {
        "imaging_modality": {"type": "string"},
        "key_findings": {
            # Resilient to both: "Lung is clear" OR ["Lung is clear", "Heart size normal"]
            "anyOf": [
                {"type": "string"},
                {
                    "type": "array", 
                    "items": {"type": "string"}
                }
            ]
        },
        "laterality": {"type": "string"},
        "critical_finding_detected": {"type": "boolean"},
        "confidence_level": {
            "type": "string", 
            "enum": ["low", "medium", "high"]
        },
        "technical_quality": {
            "type": "string", 
            "enum": ["adequate", "poor"]
        },
        "safety_note": {"type": "string"}
    },
    "required": [
        "imaging_modality", 
        "key_findings", 
        "laterality", 
        "critical_finding_detected"
    ]
}

def validate_and_repair(raw_ai_response, original_context, max_retries=2):
    """
    Ensures the Scribe Agent's output matches the SOAP_SCHEMA.
    If it fails, it re-prompts the model with the specific error[cite: 33, 46].
    """
    current_response = raw_ai_response
    
    for attempt in range(max_retries + 1):
        try:
            # 1. PARSE: Extract JSON content from potential markdown markers [cite: 44]
            start_idx = current_response.find('{')
            end_idx = current_response.rfind('}') + 1
            
            if start_idx == -1 or end_idx == 0:
                raise ValueError("No JSON object found in response.")
                
            clean_json = current_response[start_idx:end_idx]
            data = json.loads(clean_json)
            
            if original_context == "Primary Vision Analysis Task" :
                json_schema=VISION_SCHEMA
            else:
                json_schema=SOAP_SCHEMA
                
            validate(instance=data, schema=json_schema)   
                
            return data, "Success" # Perfect output!

        except (json.JSONDecodeError, jsonschema.ValidationError, ValueError) as e:
            if attempt < max_retries:
                # 3. REPAIR: Construct a targeted repair prompt [cite: 33, 46]
                error_msg = str(e)
                repair_prompt = f"""
                Your previous JSON output was invalid.
                ERROR: {error_msg}
                CONTEXT: {original_context}
                
                FIX: Please provide the corrected, flat JSON object aligned to {json_schema}. Ensure all required fields are present.
                """
                
                current_response = call_medgemma(repair_prompt) 
                
                print(f"Attempt {attempt + 1} failed. Triggering self-repair loop...")
                continue
            else:
                # 4. ESCALATE: If repairs fail, route to human review [cite: 37, 47]
                return None, f"Schema Failure after {max_retries} attempts: {str(e)}"

## Call imaging specialist agent

In [11]:
def call_imaging_specialist(vision_prompt, image_path):
    try:
        visual_findings = call_medgemma_universal(vision_prompt, image_path)
        
        # This likely returns (dict, status)
        vision_json, status = validate_and_repair(visual_findings, original_context="Primary Vision Analysis Task")

        # FIX: Even if there's an error, return a dictionary 
        # so the next agent doesn't crash!
        if isinstance(vision_json, str) and "ERROR" in vision_json:
             return {
                 "key_findings": ["Imaging analysis currently unavailable."],
                 "error": True
             }

        return vision_json

    except Exception as e:
        # FIX: Return a dict instead of an AGENT_ERROR string
        return {
            "key_findings": [f"Specialist access failure: {str(e)}"],
            "error": True
        }

## Call Scribe agent

In [12]:
def call_scribe_agent(scribe_prompt):
    
    # Run through the Gemma-2-2b text model
    response = call_medgemma_universal(scribe_prompt)
    
    # 1. Look for the very last '}' in the entire response
    end_idx = response.rfind('}') + 1
    
    # 2. Look for the '{' that corresponds to that specific block
    # We search backwards from the end_idx
    remaining_text = response[:end_idx]
    start_idx = remaining_text.rfind('{')
    
    if start_idx != -1 and end_idx > start_idx:
        clean_json = response[start_idx:end_idx]
        return clean_json
    
    return response # Fallback if logic fails

## Call Independent Auditor

In [13]:
def call_independent_auditor(auditor_input_prompt):
    try:
        # 1. Get the raw talkative response from the model
        raw_response = call_medgemma_universal(auditor_input_prompt) 

        print ("raw response from text_api: " , raw_response)

        # 2. TARGET THE FINAL JSON BLOCK
        # We find the absolute last '}' in the entire text
        end_idx = raw_response.rfind('}') + 1
        
        # We find the matching '{' by searching backwards from that '}'
        remaining_text = raw_response[:end_idx]
        start_idx = remaining_text.rfind('{')
        
        if start_idx == -1:
             return {"audit_error": ["CRITICAL: No JSON flags found in auditor response."]}
            
        clean_json = raw_response[start_idx:end_idx].strip()
        
        # 3. PARSE: Convert string to Python dictionary
        return json.loads(clean_json)
        
    except Exception as e:
        # Safety Fallback: Any error in the audit results in an escalation
        return {"audit_error": [f"AUDIT SYSTEM ERROR: {str(e)}"]}

## Risk gate and human review

In [14]:
def risk_gate_and_triage(processed_text, soap_json, warnings):
    """
    Evaluates the 'Safety Score' of the encounter.
    If risk is too high, triggers a Human Review Packet[cite: 47, 94].
    """
    risk_points = 0
    
    # 1. Privacy Risk: Too many redactions
    redaction_count = processed_text.count("[") 
    if redaction_count > 3: risk_points += 5
    
    # 2. Clinical Risk: Serious contradictions [cite: 36]
    if len(warnings) >= 2: risk_points += 10
    
    # 3. Status Risk: Critical patients need eyes 
    if soap_json.get("patient_status") == "Critical": risk_points += 5

    # THRESHOLD CHECK (e.g., 10 points = High Risk)
    if risk_points >= 10:
        return True, risk_points
    return False, risk_points

def generate_human_review_packet(soap_json, warnings, risk_score):
    """Creates a 'Safety First' report for a clinician to manually sign off[cite: 47]."""
    return f"""
    # ⚠️ MANUAL CLINICIAN REVIEW REQUIRED
    **Risk Score:** {risk_score}/20
    
    **Reason for Escalation:** System detected high clinical risk or excessive data sensitivity.
    
    ### 🚨 Audit Warnings:
    {chr(10).join(warnings)}
    
    ### 📝 Drafted SOAP (Needs Verification):
    **Assessment:** {soap_json.get('assessment')}
    **Plan:** {soap_json.get('plan')}
    
    *Action: Please verify findings and complete documentation manually.*
    """

## Scribe & Audit Loop

In [15]:
def scribe_and_audit(vision_json_str, key_terms, raw_notes, max_retries=3):

    validated_json = None
    iteration = 0
    history = []

    while iteration < max_retries:
        iteration += 1
        
        if validated_json is None:
            # ROUND 1: Initial Draft
            scribe_prompt = CLINICAL_SCRIBE_PROMPT.format(visual_findings=vision_json_str,symptoms_text=raw_notes)
        else:
            scribe_prompt= MISTAKE_EDITOR_PROMPT.format(
            rejected_soap=validated_json,
            audit_findings_str=audit_findings_str,
            vision_json_str=vision_json_str,
            symptoms_text=raw_notes
            )
        
        current_soap = call_scribe_agent(scribe_prompt)
        validated_json, status_msg = validate_and_repair(current_soap, scribe_prompt)
        
        # CRITICAL GATE: If repair fails after max retries, stop and notify the user 
        if not validated_json:
            return {"soap": "unavailable", "history": history, "success": False}

        # 3. THE AUDIT GATE
        audit_prompt = CoT_AUDITOR_PROMPT.format(
                symptoms_text=raw_notes,
                visual_findings=vision_json_str,
                soap_json=json.dumps(validated_json),
                required_findings=key_terms
        )
        audit_results = call_independent_auditor(audit_prompt)
        #flags = audit_results.get("flags", [])
                
        audit_status = audit_results.get("safety_audit", []).get("status",[])
    
        if audit_status== "CLEAN" :
            print(f"✨ Approved on Round {iteration}!")
            return {"soap": validated_json, "history": history, "success": True}

        # if audit_status is not CLEAN, and there is a valid json
        audit_findings = audit_results.get("safety_audit", []).get("audit_findings",[])
        audit_findings_str= json.dumps(flags, indent=2)
        history.append({"round": iteration, "audit_flags": audit_findings, "note": copy.deepcopy(validated_json)})

        print(f"⚠️ Round {iteration} failed. Auditor flags: {flags}")

    return {"soap": validated_json, "history": history, "success": False}

## Med Brain Squad Orchestrator

In [16]:
def med_brain_squad_orchestrator(image, symptoms_text):

    # Step 0 - initial check for text input
    if not symptoms_text or symptoms_text.strip() == "":
        return "⚠️ Please provide at least a messy note or patient history.", {}
    
    # Step A: Privacy Check 
    is_safe, processed_text = pii_deidentification_gate(symptoms_text)
    if not is_safe:
        return "❌ SAFETY ALERT: Patient PII detected. Please sanitize notes.", {}

    # --- Step B: Imaging Specialist ---
    visual_json_str = "No image provided."
    key_vision_terms = ""
    
    # Step B: Imaging Specialist Agent 
    # This role focuses ONLY on the visual findings
    if image is not None:
        visual_json = call_imaging_specialist(IMAGING_SPECIALIST_PROMPT, image)
        visual_json_str = str(visual_json)
        key_vision_terms = ", ".join(visual_json.get("key_findings", []))
        MemoryManager.clear()

    # Step C - scribe & audit loop
    result  = scribe_and_audit(visual_json_str,key_vision_terms, processed_text)
    MemoryManager.clear()
    audited_soap = result["soap"]
    history = result["history"]
    success = result["success"]

    warnings = history[-1]["audit_flags"] if history else []
        
    # Step D: Risk triage 
    is_high_risk, score = risk_gate_and_triage(processed_text, audited_soap, warnings)

    # Step G: Conditional Output (The Final Gate)
    if is_high_risk:
        # Route to human review packet 
        display_report = generate_human_review_packet(audited_soap, warnings, score)
    else:
        # Standard automated report construction
        display_report = format_standard_report(audited_soap, warnings)

    return display_report, audited_soap, history, key_vision_terms



## Display standard report

In [17]:
def format_standard_report(validated_json, warnings):
    """
    Transforms validated clinical data into a professional Markdown report.
    Prioritizes safety alerts and patient status.
    """
    # 1. Status Indicator (Visual Triage)
    status = validated_json.get("patient_status", "Unknown").upper()
    status_emoji = "🟢" if status == "STABLE" else ("🔴" if status == "CRITICAL" else "🟡")
    
    # 2. Risk Header
    risk_score = validated_json.get("risk_score", 0)
    risk_bar = "█" * risk_score + "░" * (10 - risk_score)
    
    # 3. Warning Section (Safety Gate)
    warning_section = ""
    if warnings:
        warning_list = "\n".join([f"* {w}" for w in warnings])
        warning_section = f"""
        > ### 🚨 SAFETY AUDIT ALERTS
        > {warning_list}
        > 
        > *Note: Please cross-reference findings with primary source data.*
        ---
        """
        
    # 4. Constructing the SOAP Report 
    report_md = f"""
    # {status_emoji} CLINICAL SUMMARY: {status}
    **Risk Score:** `{risk_score}/10` [{risk_bar}]
    
    {warning_section}
    
    ### 📝 SUBJECTIVE
    {validated_json.get('subjective', 'No history provided.')}
    
    ### 🔍 OBJECTIVE
    {validated_json.get('objective', 'No objective findings provided.')}
    
    ### 🧠 ASSESSMENT
    **Primary Impression:** {validated_json.get('assessment', 'Pending clinical review.')}
    
    ### 📋 PLAN
    {validated_json.get('plan', 'Awaiting clinician sign-off.')}
    
    ---
    **System Guarantees:** 🛡️ PII-Safe | ✅ Schema-Validated | ⚖️ Audit-Checked [cite: 50, 51, 52]
    """
    return report_md

## Gradio Interface

In [18]:

def gradio_interface(image, symptoms):
    # 1. Run the Full Orchestrator
    # This calls your Vision -> Scribe -> Auditor loop
    final_report, audit_data, history, vision_findings = med_brain_squad_orchestrator(image, symptoms)
    
    # 2. Format the Agent History for the Chatbot
    # We turn your 'history' list into a conversation format
    chat_history = []

    #Vision Agent entry, to  see the whole squad working
    if vision_findings:
        findings_text = "\n".join([f"- {f}" for f in vision_findings]) if isinstance(vision_findings, list) else vision_findings
        chat_history.append({
            "role": "assistant", 
            "content": f"🔭 **Vision Specialist:** Analysis complete. I detected:\n{findings_text}"
        })
    
    for entry in history:
        round_num = entry['round']
        flags = entry['flags']
        
        chat_history.append({"role": "assistant", "content": f"📝 **Scribe (Round {round_num}):** Drafted initial SOAP."})
        
        if flags:
            flag_text = "\n".join([f"- {f}" for f in flags])
            chat_history.append({"role": "user", "content": f"⚖️ **Auditor:** REJECTED. Found errors:\n{flag_text}", "metadata": {"title": "Audit Logic"}})
        else:
            chat_history.append({"role": "user", "content": "⚖️ **Auditor:** APPROVED. No clinical mismatches found."})

    return final_report, chat_history

# --- BUILD THE UI ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧠 MedBrainSquad: Agentic Clinical Assistant")
    
    with gr.Row():
        with gr.Column(scale=1):
            input_img = gr.Image(type="filepath", label="Upload Chest X-Ray")
            input_text = gr.Textbox(label="Initial Clinical Notes (Messy)", lines=5)
            submit_btn = gr.Button("🚀 Generate Verified Report", variant="primary")
            
        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem("📄 Final Verified Report"):
                    output_report = gr.Markdown(label="Final SOAP")
                with gr.TabItem("🕵️ Agent Decision Logs"):
                    output_chat = gr.Chatbot(label="Scribe-Auditor Interaction", type="messages")

    submit_btn.click(
        fn=gradio_interface, 
        inputs=[input_img, input_text], 
        outputs=[output_report, output_chat]
    )


# Close any open interfaces to free the loop
gr.close_all()

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://087c817890db9b6bf6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
